<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Extracción de datos de diferentes fuentes: Archivos SHP</h3>
    </div>
</div>


# Archivos SHP

Los archivos Shapefile (`.shp`) almacenan geometrías vectoriales, como puntos, líneas o polígonos, junto con atributos descriptivos. En realidad forman un conjunto de archivos relacionados: `.shp` guarda la geometría, `.shx` el índice espacial y `.dbf` los atributos; el `.prj` describe el sistema de coordenadas cuando está presente.


Un Shapefile no es un único archivo. Para que una capa funcione correctamente deben conservarse juntos, como mínimo:

| Extensión | Contenido |
|---|---|
| `.shp` | Geometría de puntos, líneas o polígonos |
| `.shx` | Índice de las geometrías |
| `.dbf` | Tabla de atributos |
| `.prj` | Sistema de referencia de coordenadas (CRS) |


También pueden existir archivos `.cpg`, `.sbn` y `.sbx`. Si falta alguno de los archivos principales, la capa puede no abrirse o perder información.

### GeoPandas como extensión de pandas

`geopandas` extiende `pandas` con una columna `geometry` y un sistema de referencia espacial (`crs`). 

`geopandas.GeoDataFrame` combina una tabla de atributos con una columna `geometry`. Esto permite aplicar operaciones conocidas de pandas y, además, trabajar con mapas:

- `gpd.read_file()` lee un Shapefile.
- `gdf.columns`, `gdf.head()` y `gdf.info()` inspeccionan la tabla.
- `gdf.geometry.geom_type` identifica el tipo de geometría.
- `gdf.crs` muestra el sistema de coordenadas.
- `gdf.plot()` dibuja la capa.
- `to_file()` exporta una nueva capa.
- `to_crs()` reproyecta las geometrías.
- `sjoin()` combina capas según su relación espacial.

La librería `geopandas` permite leer y manipular estos archivos de forma parecida a un DataFrame, pero manteniendo la geometría. La visualización temática del ejemplo colorea cada geometría usando una columna de atributos; esto ayuda a comprobar que la tabla y el mapa se alinean.

### Sistemas de referencia de coordenadas

El CRS indica cómo se ubican las geometrías sobre la Tierra. Un CRS geográfico como `EPSG:4326` utiliza longitud y latitud en grados. Para medir áreas o distancias conviene reproyectar a un CRS proyectado adecuado para la zona de estudio.

###  Modelo de datos vectoriales

Un archivo SHP representa objetos espaciales mediante **geometrías vectoriales**. Cada fila de un `GeoDataFrame` describe una entidad: sus columnas contienen atributos y la columna `geometry` contiene su posición y forma.

| Tipo de geometría | Qué representa | Ejemplos | Operaciones frecuentes |
|---|---|---|---|
| `Point` | Una ubicación formada por una coordenada | sensores, escuelas, casos registrados | contar, localizar, unir con polígonos |
| `LineString` | Una trayectoria con longitud | carreteras, ríos, rutas | longitud, intersección, red |
| `Polygon` | Una superficie cerrada | municipios, lagos, zonas censales | área, contiene, intersección |
| `MultiPoint` | Varias ubicaciones agrupadas | conjunto de mediciones | agrupar o separar puntos |
| `MultiLineString` | Varias líneas que forman una entidad | una red o tramo discontinuo | longitud total, unión |
| `MultiPolygon` | Varias superficies que pertenecen a una entidad | islas de un mismo municipio | área total, disolución |

### Cómo interpretar una capa
- **Geometría:** responde a “¿dónde está y qué forma tiene?”.
- **Atributos:** responden a “¿qué características tiene?”. Por ejemplo, nombre, clave, población o fecha.
- **CRS:** indica cómo interpretar las coordenadas. Sin un CRS correcto, dos capas pueden no coincidir aunque sus números parezcan válidos.

La geometría determina qué preguntas espaciales son posibles. Para una capa de puntos suele ser útil contar registros o calcular distancias; para una capa de líneas, medir longitud; para una capa de polígonos, calcular área, comparar superficies o realizar operaciones de contención e intersección.

> antes de calcular áreas o distancias, es necesario revisar `gdf.crs`. Un CRS geográfico usa grados. Un CRS proyectado adecuado para la zona suele usar metros.

## Ejemplo de extracción de datos de un archivo SHP

1. **Conservar el conjunto de archivos:** `.shp`, `.shx`, `.dbf` y `.prj` deben mantenerse juntos y con el mismo nombre base.
2. **Leer la capa:** usar `gpd.read_file()` y guardar el resultado en una variable descriptiva.
3. **Inspeccionar:** revisar dimensiones, columnas, CRS, tipos de geometría, valores nulos y límites espaciales.
4. **Limpiar y filtrar:** corregir tipos, seleccionar atributos y trabajar con una copia cuando se modifique una capa.
5. **Reproyectar cuando sea necesario:** usar `to_crs()` antes de calcular distancias o áreas.
6. **Analizar y visualizar:** elegir una operación coherente con el tipo de geometría.
7. **Exportar:** usar CSV para atributos tabulares y GeoPackage o SHP para conservar geometrías.


In [ ]:
# Configuración común para ejecutar esta sección de forma independiente
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


dir_base = os.getcwd()
print(dir_base)
ruta = os.path.join(dir_base, 'Data')
ruta_covid_india = os.path.join(dir_base, 'COVID_INDIA_POC-shp')
print(ruta)
print(ruta_covid_india)

In [ ]:
# %conda !pip !conda
#%pip install geopandas

In [ ]:
import geopandas as gpd

La capa `COVID_INDIA_POC.shp` contiene puntos de registros asociados con ubicaciones de India. Cada fila combina atributos descriptivos, como estado y distrito, con una geometría espacial de tipo `Point`.

In [ ]:
#`read_file()` devuelve un `GeoDataFrame`. Además de las columnas de atributos, contiene `geometry`, que almacena la ubicación de cada registro.
gdf_covid = gpd.read_file(os.path.join(ruta_covid_india, 'COVID_INDIA_POC.shp'))
gdf_covid.head()

In [ ]:
def resumen_geografico(gdf):
    return pd.Series({
        'filas': len(gdf),
        'columnas': len(gdf.columns),
        'crs': str(gdf.crs),
        'geometrias': gdf.geometry.geom_type.value_counts().to_dict(),
        'geometrias_vacias': int(gdf.geometry.is_empty.sum()),
        'geometrias_nulas': int(gdf.geometry.isna().sum()),
        'geometrias_invalidas': int((~gdf.geometry.is_valid).sum()),
    })

resumen_geografico(gdf_covid)

In [ ]:
# Leer e inspeccionar una capa
print(gdf_covid.shape)
print(gdf_covid.columns.tolist())
print(gdf_covid.crs)
gdf_covid.head()


In [ ]:
# Consultar geometrías y CRS
print(gdf_covid.geometry.geom_type.value_counts())
print('CRS:', gdf_covid.crs)
print('Límites:', gdf_covid.total_bounds)
print('Geometrías vacías:', gdf_covid.geometry.is_empty.sum())
print('Geometrías inválidas:', (~gdf_covid.geometry.is_valid).sum())


### Operaciones según el tipo de geometría

Las propiedades geométricas se consultan desde `gdf.geometry`. Algunas operaciones comunes son:

| Pregunta | Expresión típica | Unidad o resultado |
|---|---|---|
| ¿Qué tipo de objeto hay? | `gdf.geometry.geom_type` | `Point`, `LineString`, `Polygon`, etc. |
| ¿Cuál es su extensión? | `gdf.total_bounds` | `xmin, ymin, xmax, ymax` |
| ¿Cuánto mide una línea? | `gdf.geometry.length` | unidades del CRS |
| ¿Qué superficie ocupa un polígono? | `gdf.geometry.area` | unidades cuadradas del CRS |
| ¿Está dentro de otra geometría? | `gdf.geometry.within(otra)` | valores booleanos |
| ¿Se intersectan dos capas? | `gpd.sjoin(...)` | unión espacial de filas |

`length` y `area` no convierten unidades por sí mismas. Si el CRS está en grados, el resultado estará en grados o grados cuadrados y no será una medición física útil.

In [ ]:
# Operaciones geométricas básicas para la capa de puntos
tipos_geometria = gdf_covid.geometry.geom_type.value_counts()
print('Tipos encontrados:')
display(tipos_geometria.to_frame('cantidad'))

# En una capa de puntos, cada geometría tiene coordenadas x e y.
if (gdf_covid.geometry.geom_type == 'Point').all():
    coordenadas = gdf_covid.assign(
        longitud=gdf_covid.geometry.x,
        latitud=gdf_covid.geometry.y,
    )
    display(coordenadas[['State_UT', 'District', 'longitud', 'latitud']].head())

# Para líneas se usaría geometry.length y para polígonos geometry.area,
# siempre después de revisar o cambiar el CRS cuando necesites unidades físicas.

La capa utiliza `EPSG:4326`, un CRS geográfico basado en longitud y latitud. Es adecuado para ubicar los puntos, pero no para calcular directamente distancias o áreas en metros.

In [ ]:
# Filtrar y resumir atributos
# Registros de un estado
registros_maharashtra = gdf_covid[
    gdf_covid['State_UT'].eq('MAHARASHTRA')
].copy()
print(registros_maharashtra)


In [ ]:
# Número de registros por estado
por_estado = (
    gdf_covid['State_UT']
    .value_counts()
    .rename_axis('estado')
    .reset_index(name='registros')
)

In [ ]:
# Número de distritos por estado
por_estado['distritos'] = por_estado['estado'].map(
    gdf_covid.groupby('State_UT')['District'].nunique()
)
por_estado.head()


In [ ]:
# Reproyectar para medir distancias
# proyección apropiada para la región de estudio.
gdf_metros = gdf_covid.to_crs('EPSG:3857')


In [ ]:
# Distancia desde el centro de la extensión espacial de la capa.

punto_referencia = gdf_metros.geometry.union_all().centroid
gdf_metros['distancia_m'] = gdf_metros.geometry.distance(punto_referencia)
gdf_metros[['State_UT', 'District', 'distancia_m']].head()

In [ ]:
# Visualización general de los registros
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 8))
gdf_covid.plot(
    ax=ax,
    color='#0f766e',
    markersize=12,
    alpha=0.65,
    edgecolor='white',
    linewidth=0.3,
)
ax.set_title('Registros COVID por ubicación')
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.grid(alpha=0.25)
plt.show()



In [ ]:
# Mapa temático por estado
fig, ax = plt.subplots(figsize=(13, 9))
gdf_covid.plot(
    ax=ax,
    column='State_UT',
    categorical=True,
    cmap='tab20',
    markersize=14,
    alpha=0.75,
    legend=True,
    legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
)
ax.set_title('Registros COVID por estado o territorio de India')
ax.set_axis_off()
plt.tight_layout()
plt.show()

Como esta capa contiene puntos, el mapa representa ubicaciones de registros y no fronteras administrativas. Para mostrar límites estatales se necesita otra capa poligonal compatible.


In [ ]:
# Visualizar un estado específico

estado = 'MAHARASHTRA'
gdf_estado = gdf_covid[gdf_covid['State_UT'].eq(estado)]

fig, ax = plt.subplots(figsize=(10, 7))
gdf_estado.plot(
    ax=ax,
    column='District',
    categorical=True,
    cmap='Set3',
    markersize=22,
    legend=True,
)
ax.set_title(f'Registros COVID en {estado}')
ax.set_axis_off()
plt.show()

In [ ]:
### Exportar resultados
por_estado.to_csv('covid_resumen_estados.csv', index=False, encoding='utf-8-sig')
gdf_covid.to_file('covid_india_procesado.gpkg', layer='registros', driver='GPKG')

El CSV conserva atributos tabulares, mientras que GeoPackage conserva atributos y geometría. Para no perder la ubicación espacial, se recomienda no exportar únicamente a CSV si se necesita continuar trabajando con mapas.

## Práctica de Laboratorio: Análisis geostadístico de los municipios de Jalisco

Descarga una capa de municipios de Jalisco desde el INEGI [Marco Geoestadístico municipal 1995. Conteo de Población y Vivienda 1995](https://www.inegi.org.mx/app/biblioteca/ficha.html?upc=702825292836) guárdala en una carpeta Data en tu repositorio de clase. El archivo principal debe tener extensión `.shp` y sus archivos auxiliares deben permanecer en la misma carpeta.

El objectivo de esta actividad es: Leer una capa geográfica con GeoPandas, inspeccionar sus atributos y geometrías, construir un mapa temático y exportar resultados tabulares para su análisis posterior.

### Instrucciones

1. Carga la capa con `gpd.read_file()`.
2. Inspecciona `head()`, `columns`, `shape`, `crs` y `geometry.geom_type`.
3. Identifica la columna que contiene el nombre o clave del municipio.
4. Calcula el área de cada municipio. Antes de calcularla, reproyecta la capa a un CRS proyectado apropiado para Jalisco. (Hint: use el atributo `geometry.area` del CRS proyectado)
5. Ordena los municipios por superficie y muestra los cinco más grandes.
6. Crea un mapa de los municipios utilizando `plot()`, bordes visibles y una escala de color basada en el área.
7. Exporta una tabla sin geometría a `municipios_jalisco.csv`.



In [ ]:
# 1. Carga la capa con `gpd.read_file()`.


In [ ]:
# 2. Inspeccionar la capa


In [ ]:
# 3. Identifica la columna que contiene el nombre o clave del municipio.


In [ ]:
# 4. Calcula el área de cada municipio. Antes de calcularla, reproyecta la capa a un CRS proyectado apropiado para Jalisco.
# Filtrar únicamente los municipios de Jalisco


In [ ]:
# Reproyectar Jalisco a un CRS proyectado


In [ ]:
# Calcular el área de cada municipio en km²


In [ ]:
#5. Ordena los municipios por superficie y muestra los cinco más grandes.


In [ ]:
# 6. Crea un mapa de los municipios utilizando `plot()`, bordes visibles y una escala de color basada en el área.


In [ ]:
# 7. Exporta una tabla sin geometría a `municipios_jalisco.csv`.
